In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import json
from pathlib import Path

In [3]:
# Set path of the green and yellow schema files
green_schema_path = Path("../schemas/green_tripdata.schema.json")
yellow_schema_path = Path("../schemas/yellow_tripdata.schema.json")

In [4]:
spark = (SparkSession
         .builder
         .master("local[*]")
         .appName("taxi_conversion")
         .getOrCreate()
         )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 20:19:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# infer schema for green trip data
df_green_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/raw/green/2020/01")

In [6]:
df_green_inferred.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [7]:
# save green trip schema to json
green_schema_json = df_green_inferred.schema.json()

In [8]:
# Create green schema path if it does not exist:
if not green_schema_path.parent.exists():
    green_schema_path.parent.mkdir(parents=True, exist_ok=True)

# pretty-print the JSON
green_schema_pretty = json.loads(green_schema_json)

with green_schema_path.open("w") as f:
    json.dump(green_schema_pretty, f, indent=2)

print(f"Green trip schema saved to {green_schema_path.resolve()}")





Green trip schema saved to /Users/mcopple/src/zoomcamps/src/de-homework-2026/module-6/examples/schemas/green_tripdata.schema.json


In [20]:
df_yellow_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/raw/yellow/2020/01")

yellow_schema_json = df_yellow_inferred.schema.json()

# Create yellow schema path if it does not exist:
if not yellow_schema_path.parent.exists():
    yellow_schema_path.parent.mkdir(parents=True, exist_ok=True)

# pretty-print the JSON
yellow_schema_pretty = json.loads(yellow_schema_json)

with yellow_schema_path.open("w") as f:
    json.dump(yellow_schema_pretty, f, indent=2)

print(f"Yellow trip schema saved to {yellow_schema_path.resolve()}")


Yellow trip schema saved to /Users/mcopple/src/zoomcamps/src/de-homework-2026/module-6/examples/schemas/yellow_tripdata.schema.json


In [11]:
# open green trip data
with open("../schemas/green_tripdata.schema.json") as f:
    green_schema_dict = json.load(f)

green_schema = StructType.fromJson(green_schema_dict)

df_green = spark.read \
    .option("header", True) \
    .schema(green_schema) \
    .csv("../data/raw/green/2021/01/")

In [12]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [21]:
# open yellow trip data
with open("../schemas/yellow_tripdata.schema.json") as f:
    yellow_schema_dict = json.load(f)

yellow_schema = StructType.fromJson(yellow_schema_dict)

df_yellow = spark.read \
    .option("header", True) \
    .schema(yellow_schema) \
    .csv("../data/raw/yellow/2021/01/")

In [22]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [23]:
# Write trip data to parquet
for color in ["green", "yellow"]:
    if color == "green":
        schema = green_schema
    else:
        schema = yellow_schema
    for year in range(2020, 2022):
        for month in range(1, 13):
            month_str = f"{month:02d}"
            input_path = f"../data/raw/{color}/{year}/{month_str}/"
            output_path = f"../data/pq/{color}/{year}/{month_str}/"

            df = spark.read \
                .option("header", True) \
                .schema(schema) \
                .csv(input_path)

            df.repartition(10).write.parquet(output_path)
            print(f"{color.capitalize()} trip data for {year}-{month_str} written to {output_path}")

26/03/05 20:57:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/05 20:57:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/05 20:57:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Green trip data for 2020-01 written to ../data/pq/green/2020/01/


26/03/05 20:57:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/05 20:57:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/05 20:57:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Green trip data for 2020-02 written to ../data/pq/green/2020/02/


26/03/05 20:57:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/05 20:57:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/05 20:57:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:57:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


...

Yellow trip data for 2021-05 written to ../data/pq/yellow/2021/05/


26/03/05 20:58:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/05 20:58:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:58:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/05 20:58:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:58:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Yellow trip data for 2021-06 written to ../data/pq/yellow/2021/06/


26/03/05 20:58:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/05 20:58:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:58:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/05 20:58:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/05 20:58:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Yellow trip data for 2021-07 written to ../data/pq/yellow/2021/07/
Yellow trip data for 2021-08 written to ../data/pq/yellow/2021/08/
Yellow trip data for 2021-09 written to ../data/pq/yellow/2021/09/
Yellow trip data for 2021-10 written to ../data/pq/yellow/2021/10/
Yellow trip data for 2021-11 written to ../data/pq/yellow/2021/11/
Yellow trip data for 2021-12 written to ../data/pq/yellow/2021/12/
